# STED-Inspired Depletion Beam: Activation Steering for Subspace Collapse
*Tests whether the typosquat probe direction is causally necessary for detection.*

In [ ]:
!pip install -q torch transformers scikit-learn matplotlib seaborn

In [ ]:
import json, random, re, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# Upload dataset
from google.colab import files
import zipfile, io
uploaded = files.upload()
fname = list(uploaded.keys())[0]

if fname.endswith('.zip'):
    with zipfile.ZipFile(io.BytesIO(uploaded[fname])) as zf:
        jsonl = [n for n in zf.namelist() if n.endswith('.jsonl')][0]
        zf.extract(jsonl)
        DATASET_PATH = jsonl
else:
    DATASET_PATH = fname
print(f"Using: {DATASET_PATH}")

In [ ]:
df = pd.DataFrame([json.loads(line) for line in open(DATASET_PATH)])
df = df[df['is_adversarial'] == True].copy()
print(f"Loaded {len(df)} adversarial entries")

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto", device_map="auto" if device.type=="cuda" else None)
if device.type != "cuda": model.to(device)
model.eval()

In [ ]:
def find_package_span(cmd, pkg):
    m = re.search(r'\b' + re.escape(pkg) + r'\b', cmd)
    return (m.start(), m.end()) if m else None

def char_to_token_span(tokenizer, text, cs, ce):
    enc = tokenizer(text, return_offsets_mapping=True, add_special_tokens=True)
    offs = enc['offset_mapping']
    ts = te = None
    for i, (s, e) in enumerate(offs):
        if s <= cs < e or s < ce <= e:
            if ts is None: ts = i
            te = i
    return (ts or len(offs)-1, te or len(offs)-1)

def extract_hidden_states(model, tokenizer, commands, packages, layers=(12,16), batch_size=8):
    model.eval()
    all_states = []
    for i in range(0, len(commands), batch_size):
        batch_cmds = commands[i:i+batch_size]
        batch_pkgs = packages[i:i+batch_size]
        inputs = tokenizer(batch_cmds, return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            hidden = outputs.hidden_states
        for j, (cmd, pkg) in enumerate(zip(batch_cmds, batch_pkgs)):
            span = find_package_span(cmd, pkg)
            if span is None:
                states = [hidden[l][j, -1, :].float().cpu().numpy() for l in range(layers[0], min(layers[1]+1, len(hidden)))]
            else:
                ts, te = char_to_token_span(tokenizer, cmd, *span)
                if ts > te or ts >= hidden[0].shape[1]:
                    states = [hidden[l][j, -1, :].float().cpu().numpy() for l in range(layers[0], min(layers[1]+1, len(hidden)))]
                else:
                    states = [hidden[l][j, ts:te+1, :].float().cpu().numpy().mean(axis=0) for l in range(layers[0], min(layers[1]+1, len(hidden)))]
            if states:
                combined = np.concatenate(states)
                combined = np.nan_to_num(combined, nan=0.0, posinf=0.0, neginf=0.0)
                all_states.append(combined)
    return np.array(all_states) if all_states else None

In [ ]:
texts, pkgs, labels = [], [], []
for _, row in df.iterrows():
    texts.append(row['clean_command']); pkgs.append(row['package_name']); labels.append(0)
    texts.append(row['typo_command']); pkgs.append(row['typo_package']); labels.append(1)

print("Extracting features...")
X_base = extract_hidden_states(model, tokenizer, texts, pkgs)
y = np.array(labels)

In [ ]:
probe = LogisticRegression(max_iter=1000, random_state=SEED)
probe.fit(X_base, y)
w_probe = probe.coef_[0]
auc_base = roc_auc_score(y, probe.predict_proba(X_base)[:, 1])
print(f"Baseline AUC: {auc_base:.4f}")

In [ ]:
def steer_activations(X: np.ndarray, w: np.ndarray, alpha: float) -> np.ndarray:
    proj = X @ w
    return X - alpha * np.outer(proj, w)

alphas = np.linspace(0, 5, 11)
results = []

for α in alphas:
    X_steered = steer_activations(X_base, w_probe, α)
    probe_s = LogisticRegression(max_iter=1000, random_state=SEED)
    probe_s.fit(X_steered, y)
    auc_s = roc_auc_score(y, probe_s.predict_proba(X_steered)[:, 1])
    w_s = probe_s.coef_[0]
    cos_sim = np.dot(w_probe, w_s) / (np.linalg.norm(w_probe) * np.linalg.norm(w_s))
    results.append({'alpha': α, 'auc': auc_s, 'cosine_similarity': cos_sim})
    print(f"α={α:.1f}: AUC={auc_s:.4f}, cosine={cos_sim:.4f}")

In [ ]:
res_df = pd.DataFrame(results)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,5))
sns.lineplot(data=res_df, x='alpha', y='auc', marker='o', ax=ax1)
ax1.axhline(0.5, ls='--', c='gray', label='Chance')
ax1.set_xlabel('Steering Strength α'); ax1.set_ylabel('Probe AUC')
ax1.set_title('Depletion Beam: Subspace Collapse')
ax1.legend(); ax1.grid(alpha=0.3)

sns.lineplot(data=res_df, x='alpha', y='cosine_similarity', marker='s', color='orange', ax=ax2)
ax2.set_xlabel('Steering Strength α'); ax2.set_ylabel('Direction Cosine')
ax2.set_title('Probe Rotation Under Steering')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('sted_depletion_curve.png', dpi=150)
plt.show()

In [ ]:
collapse_thresh = 0.7
collapsed = res_df[res_df['auc'] < collapse_thresh]
if not collapsed.empty:
    α_min = collapsed['alpha'].min()
    print(f"✓ Subspace collapses at α ≥ {α_min:.2f}")
else:
    print(f"⚠ AUC remains ≥ {collapse_thresh} up to α=5")
    print("  → Try contrastive fine-tuning or multi‑direction steering.")